# 部署

当你准备好把 LangGraph agent 部署到生产环境时, 请选择一种适合你的技术栈的托管模式。**[LangSmith Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud)** 为有状态、长时运行的 agent 提供全托管基础设施, 具备持久 state 与后台执行能力。

> **提示**
>
> 除了 Cloud 之外, LangSmith 还提供多种部署选项, 包括 [hybrid](https://docs.langchain.com/langsmith/hybrid)、[standalone servers](https://docs.langchain.com/langsmith/deploy-standalone-server) 以及 [self-hosted with control plane](https://docs.langchain.com/langsmith/deploy-with-control-plane)。更多信息见 [LangSmith Deployment 概览](https://docs.langchain.com/langsmith/deployment)。

In [1]:
# 环境准备: 从 .env 加载 API keys
from dotenv import load_dotenv

load_dotenv()  # 自动向上查找仓库根目录的 .env

# ---- 统一导入 ----

True

In [2]:
from IPython.display import display, Image
from langgraph.graph.state import CompiledStateGraph

def show_graph(graph: CompiledStateGraph) -> None:
    display(Image(graph.get_graph().draw_mermaid_png()))

from rich import print as rprint


## LangSmith Cloud

本节将带你从 GitHub 仓库把 agent 部署到 LangSmith Cloud。LangSmith 负责基础设施、扩展与运维相关事项。

### 前置条件

在开始之前, 请确保你具备以下条件:

- 一个 [GitHub 账号](https://github.com/)
- 一个 [LangSmith 账号](https://smith.langchain.com)

### 部署你的 agent

#### 1. 在 GitHub 上创建仓库

你的应用代码必须位于 GitHub 仓库中, 才能部署到 LangSmith。公有仓库与私有仓库都受支持。在本快速入门中, 请先按照 [本地服务器设置指南](https://docs.langchain.com/oss/langgraph/studio#set-up-local-agent-server) 确认你的应用与 LangGraph 兼容。然后, 把你的代码推送到该仓库。

#### 2. 部署到 LangSmith

1. **导航到 LangSmith Deployment**: 登录 [LangSmith](https://smith.langchain.com)。在左侧边栏中, 选择 **Deployments**。
2. **创建新部署**: 点击 **+ New Deployment** 按钮。会打开一个面板, 你可以在其中填写必填字段。
3. **关联仓库**: 如果你是首次使用, 或者要添加一个此前从未连接过的私有仓库, 请点击 **Add new account** 按钮, 并按照说明连接你的 GitHub 账号。
4. **部署仓库**: 选择你的应用仓库。点击 **Submit** 进行部署。这可能需要约 15 分钟才能完成。你可以在 **Deployment details** 视图中查看状态。

#### 3. 在 Studio 中测试你的应用

应用部署完成后:

1. 选择你刚刚创建的部署, 以查看更多详情。
2. 点击右上角的 **Studio** 按钮。Studio 会打开并显示你的 graph。

#### 4. 获取部署的 API URL

1. 在 LangGraph 的 **Deployment details** 视图中, 点击 **API URL** 将其复制到剪贴板。
2. 点击 `URL` 将其复制到剪贴板。

#### 5. 测试 API

现在你可以测试 API:

**Python**

先安装 LangGraph SDK:

```shell
pip install langgraph-sdk
```

然后向 agent 发送一条消息:

```python
from langgraph_sdk import get_sync_client # or get_client for async

client = get_sync_client(url="your-deployment-url", api_key="your-langsmith-api-key")

for chunk in client.runs.stream(
    None,    # Threadless run
    "agent", # Name of agent. Defined in langgraph.json.
    input={
        "messages": [{
            "role": "human",
            "content": "What is LangGraph?",
        }],
    },
    stream_mode="updates",
):
    print(f"Receiving new event of type: {chunk.event}...")
    print(chunk.data)
    print("\n\n")
```

> **说明**
>
> 上面的片段需要用真实的部署 URL 与 LangSmith API key 替换 `your-deployment-url` 与 `your-langsmith-api-key`, 它依赖一个已经部署好的实例, 因此在本文档中保留为代码块而不参与执行。

**Rest API**

```bash
curl -s --request POST \
    --url <DEPLOYMENT_URL>/runs/stream \
    --header 'Content-Type: application/json' \
    --header "X-Api-Key: <LANGSMITH API KEY> \
    --data "{
        \"assistant_id\": \"agent\", `# Name of agent. Defined in langgraph.json.`
        \"input\": {
            \"messages\": [
                {
                    \"role\": \"human\",
                    \"content\": \"What is LangGraph?\"
                }
            ]
        },
        \"stream_mode\": \"updates\"
    }"
```